# Modelo Supervisado: Clasificacion de Riesgo Crediticio
## Proyecto Final - Data Science YPF

- Se entrena un modelo de clasificacion para predecir `SeriousDlqin2yrs` a partir del dataset limpio generado en `01_analisis_exploratorio.ipynb`. 
- Se comparan varios algoritmos, se ajustan hiperparametros con `GridSearchCV` para evitar overfitting, y se evalua con metricas apropiadas para un problema desbalanceado (precision, recall, f1, ROC-AUC), no solo accuracy.
- Todo el preprocesamiento que aprende de los datos (la imputacion de faltantes con la mediana) va dentro de un `Pipeline`, de modo que se ajusta solo con train y, dentro de la validacion cruzada, solo con la parte de entrenamiento de cada fold. El set de test se usa una unica vez, al final.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    cross_val_predict,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
    f1_score,
    precision_score,
    recall_score,
    fbeta_score,
    average_precision_score,
)
from sklearn.calibration import CalibratedClassifierCV

sns.set_theme(style="whitegrid")


## 1. Carga de datos limpios

In [ ]:
df = pd.read_csv("../dataset/cs-training-clean.csv")
X = df.drop(columns="SeriousDlqin2yrs")
y = df["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(3))

## 2. Modelo base (Dummy)

Antes de comparar modelos reales, se establece un piso con `DummyClassifier`: predice siempre la clase mayoritaria. Cualquier modelo real tiene que superar esto, sobre todo en recall/f1/AUC (en accuracy el Dummy ya da ~93% por el desbalanceo).

In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
print(classification_report(y_test, y_pred_dummy, zero_division=0))

## 3. Comparacion de modelos base

In [ ]:
resultados = {}

def evaluar_modelo(nombre, modelo, X_tr, X_te, umbral=0.5):
    modelo.fit(X_tr, y_train)
    y_pred = modelo.predict(X_te)
    y_proba = modelo.predict_proba(X_te)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    y_pred_umbral = (y_proba >= umbral).astype(int)
    precision = precision_score(y_test, y_pred_umbral, zero_division=0)
    recall = recall_score(y_test, y_pred_umbral, zero_division=0)
    f2 = fbeta_score(y_test, y_pred_umbral, beta=2, zero_division=0)
    f1 = f1_score(y_test, y_pred)

    resultados[nombre] = {
        "AUC": auc,
        "PR_AUC": pr_auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "F2": f2,
        "Umbral": umbral,
    }

    print(f"--- {nombre} ---")
    print(classification_report(y_test, y_pred, zero_division=0))
    print(f"ROC-AUC: {auc:.4f}")
    print(f"PR-AUC: {pr_auc:.4f}")
    print(f"Precision (umbral {umbral}): {precision:.4f}")
    print(f"Recall (umbral {umbral}): {recall:.4f}")
    print(f"F2 (umbral {umbral}): {f2:.4f}")
    print()
    return modelo


In [ ]:
log_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
log_reg = evaluar_modelo("Regresion Logistica", log_reg, X_train, X_test)

In [ ]:
arbol = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42)),
])
arbol = evaluar_modelo("Arbol de Decision", arbol, X_train, X_test)

In [ ]:
bosque = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)),
])
bosque = evaluar_modelo("Random Forest", bosque, X_train, X_test)

In [ ]:
# Busqueda de umbral para priorizar recall y F2
def buscar_umbral_mejor(y_proba, y, metric="f2"):
    thresholds = np.linspace(0.1, 0.9, 81)
    resultados_umbral = []
    for thr in thresholds:
        pred = (y_proba >= thr).astype(int)
        resultados_umbral.append({
            "threshold": thr,
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f2": fbeta_score(y, pred, beta=2, zero_division=0),
        })
    df_umbral = pd.DataFrame(resultados_umbral)
    if metric == "f2":
        mejor = df_umbral.sort_values("f2", ascending=False).iloc[0]
    elif metric == "recall":
        mejor = df_umbral.sort_values(["recall", "precision"], ascending=[False, False]).iloc[0]
    else:
        mejor = df_umbral.sort_values("precision", ascending=False).iloc[0]
    print(df_umbral.sort_values("f2", ascending=False).head())
    print(f"Mejor umbral para F2: {mejor['threshold']:.2f} | precision={mejor['precision']:.3f}, "
          f"recall={mejor['recall']:.3f}, F2={mejor['f2']:.3f}")
    return mejor["threshold"]

## 4. Ajuste de hiperparametros (GridSearchCV)

- El Random Forest es el modelo mas prometedor. Se ajustan `n_estimators`, `max_depth` y `min_samples_leaf` con validacion cruzada (`cv=5`) optimizando ROC-AUC. Limitar la profundidad y exigir un minimo de muestras por hoja es lo que evita el overfitting: un arbol sin tope de profundidad memoriza el set de entrenamiento y generaliza peor.
- Todo el preprocesamiento que aprende de los datos (la imputacion de faltantes con la mediana) va dentro de un `Pipeline`, de modo que se ajusta solo con train y, dentro de la validacion cruzada, solo con la parte de entrenamiento de cada fold. El set de test se usa una unica vez, al final.

In [ ]:
pipe_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
])

param_grid = {
    "rf__n_estimators": [200, 400],
    "rf__max_depth": [6, 10, None],
    "rf__min_samples_leaf": [1, 5, 20],
}

grid = GridSearchCV(pipe_rf, param_grid=param_grid, scoring="roc_auc", cv=5, n_jobs=-1)
grid.fit(X_train, y_train)
print("Mejores hiperparametros:", grid.best_params_)
print("Mejor AUC en validacion cruzada:", round(grid.best_score_, 4))

## 5. Eleccion del umbral de decision

El modelo devuelve una probabilidad y el umbral define a partir de que valor se clasifica a un cliente como moroso. Con `class_weight="balanced"` y priorizando el recall, el umbral optimo no tiene por que ser 0.5.

El umbral es una decision mas del modelo, igual que los hiperparametros, por lo que **no se elige con el test**. Se obtienen probabilidades "fuera de fold" sobre train con `cross_val_predict` (cada fila recibe la prediccion de un modelo que no la vio al entrenar), se elige el umbral que maximiza F2 y recien despues se evalua en test, una sola vez.

Se usa F2 porque en scoring crediticio no detectar a un cliente que va a caer en mora (falso negativo) cuesta mas que rechazar a un buen cliente (falso positivo). F2 le da el doble de peso al recall que a la precision.

In [ ]:
mejor_modelo = grid.best_estimator_

# 1) Elegir el umbral SOLO con train, mediante validacion cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba_oof = cross_val_predict(
    mejor_modelo, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]
umbral_optimo = buscar_umbral_mejor(proba_oof, y_train, metric="f2")

# 2) Evaluar en TEST una sola vez, con el umbral ya fijo
y_proba = mejor_modelo.predict_proba(X_test)[:, 1]
y_pred = mejor_modelo.predict(X_test)                      # umbral 0,5 (solo para F1)
y_pred_ajustado = (y_proba >= umbral_optimo).astype(int)   # umbral elegido por CV

print(classification_report(y_test, y_pred_ajustado, zero_division=0))
auc_final = roc_auc_score(y_test, y_proba)
pr_auc_final = average_precision_score(y_test, y_proba)
f1_final = f1_score(y_test, y_pred)
precision_final = precision_score(y_test, y_pred_ajustado, zero_division=0)
recall_final = recall_score(y_test, y_pred_ajustado, zero_division=0)
f2_final = fbeta_score(y_test, y_pred_ajustado, beta=2, zero_division=0)

resultados["Random Forest (GridSearchCV)"] = {
    "AUC": auc_final,
    "PR_AUC": pr_auc_final,
    "Precision": precision_final,
    "Recall": recall_final,
    "F1": f1_final,
    "F2": f2_final,
    "Umbral": umbral_optimo,
}
print(f"Umbral elegido por CV en train: {umbral_optimo:.2f}")
print(f"ROC-AUC en test: {auc_final:.3f}")
print(f"PR-AUC en test: {pr_auc_final:.3f}")
print(f"Precision con umbral optimo: {precision_final:.3f}")
print(f"Recall con umbral optimo: {recall_final:.3f}")
print(f"F2 con umbral optimo: {f2_final:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_ajustado, ax=axes[0], cmap="Blues")
axes[0].set_title(f"Matriz de confusion - umbral {umbral_optimo:.2f}")
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title("Curva ROC")
plt.tight_layout()
plt.show()

In [ ]:
tabla_resultados = pd.DataFrame(resultados).T
tabla_resultados = tabla_resultados.sort_values(["F2", "Recall", "AUC"], ascending=[False, False, False])
display(tabla_resultados.round(4))

tabla_resultados["AUC"].plot(kind="barh", figsize=(7, 4), title="ROC-AUC por modelo")
plt.xlabel("ROC-AUC")
plt.tight_layout()
plt.show()

In [ ]:
importancias = pd.Series(mejor_modelo.named_steps["rf"].feature_importances_, index=X.columns)
importancias.sort_values().plot(kind="barh", figsize=(7, 6), title="Importancia de variables - Random Forest")
plt.tight_layout()
plt.show()

## 6. Conclusiones

- El Dummy logra ~93% de accuracy prediciendo siempre "no mora": confirma que la accuracy es enganosa con clases desbalanceadas. Por eso se usan ROC-AUC y PR-AUC para comparar modelos, y F2 para elegir el umbral.
- El Random Forest ajustado con `GridSearchCV` supera a la Regresion Logistica y al Arbol de Decision en ROC-AUC ([0.XXX] en test, consistente con [0.XXX] en validacion cruzada). Los mejores hiperparametros fueron `n_estimators=[...]`, `max_depth=[...]` y `min_samples_leaf=[...]`; el minimo de muestras por hoja es lo que controla el overfitting frente al Random Forest sin ajustar.
- El umbral se eligio con validacion cruzada sobre train ([0.XX]) y el test se uso una sola vez. Con ese umbral, en test: precision [0.XXX], recall [0.XXX], F2 [0.XXX]. La curva de F2 es casi plana entre 0.53 y 0.57, por lo que el valor exacto del umbral tiene poco impacto en F2.
- Las variables mas importantes son el historial de atrasos (`NumberOfTimes90DaysLate`, `NumberOfTime30-59DaysPastDueNotWorse`) y la utilizacion de credito revolvente, en linea con el AED.
- Para un uso real de scoring crediticio, el foco esta en el recall de la clase de riesgo (detectar la mayor cantidad posible de futuros morosos), aun a costa de precision. El valor exacto del umbral deberia definirse con el costo real de cada tipo de error.